# AB compartment overlap of splicing-significant genes

Goal: Based on Hi-C A/B compartment segments (e.g., stableA, stableB, A2B, B2A), calculate the proportion of genes associated with rMATS significant alternative splicing events (deduplicated) that fall into each compartment, and visualize the results.

In [ ]:
# ========== Parameters ==========
from pathlib import Path

SIG_DIR = Path("/path/to/rmat/output/significant")
SIG_GLOB = "*.sig.tsv"  # Read all event type significant results from this directory

COMPARTMENT_FILES = {
    "StableA": "/path/to/hic/stableA_compartment.bed",
    "StableB": "/path/to/hic/stableB_compartment.bed",
    "A2B": "/path/to/hic/A2B_compartment.bed",
    "B2A": "/path/to/hic/B2A_compartment.bed",
}

# Unified group order and color palette (ensures consistency across plots)
COMPARTMENT_ORDER = ["StableA", "StableB", "A2B", "B2A"]
COMPARTMENT_COLORS = {
    # Colors chosen for high distinguishability and colorblind friendliness
    "StableA": "#D55E00",     # orange-red
    "StableB": "#56B4E9",     # cyan-blue
    "A2B": "#009E73",         # green
    "B2A": "#0072B2",         # blue
}

OUTPUT_DIR = Path("/path/to/rmat/output/ab_compartment_overlap")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Parameter setup complete.")

In [ ]:
# ========== Imports & utility functions ==========
import math
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

sns.set(style="whitegrid", context="notebook")


def _coerce_int(s):
    try:
        return int(s)
    except Exception:
        try:
            return int(float(s))
        except Exception:
            return np.nan


def read_compartment_bed(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", header=None, names=["chrom", "start", "end", "label"], dtype={0: str})
    df["start"] = df["start"].map(_coerce_int)
    df["end"] = df["end"].map(_coerce_int)
    df = df.dropna(subset=["start", "end"]).copy()
    df["start"] = df["start"].astype(int)
    df["end"] = df["end"].astype(int)
    return df


def build_interval_index(df: pd.DataFrame):
    """Build an index of label -> chrom -> (starts, ends) for fast overlap computation."""
    index = {}
    for label, g1 in df.groupby("label"):
        chrom_map = {}
        for chrom, g2 in g1.groupby("chrom"):
            g2 = g2.sort_values("start")
            chrom_map[chrom] = (
                g2["start"].to_numpy(dtype=np.int64),
                g2["end"].to_numpy(dtype=np.int64),
            )
        index[label] = chrom_map
    return index


def overlapped_bp(starts: np.ndarray, ends: np.ndarray, qstart: int, qend: int) -> int:
    """Calculate the cumulative overlap length between interval [qstart, qend) and a set of non-overlapping, sorted intervals."""
    if starts.size == 0:
        return 0
    # Binary search: start scanning from the first interval that could overlap
    import bisect

    i = bisect.bisect_left(starts.tolist(), qstart)  # may fall on i-1 or i
    if i > 0:
        i -= 1
    total = 0
    n = len(starts)
    while i < n and starts[i] < qend:
        s = starts[i]
        e = ends[i]
        if e <= qstart:
            i += 1
            continue
        if s >= qend:
            break
        total += max(0, min(e, qend) - max(s, qstart))
        i += 1
    return int(total)


def assign_compartment(chrom: str, start: int, end: int, idx: dict) -> tuple:
    """Return (best_label, overlaps_dict); if no overlap, best_label = 'Unassigned'"""
    best_label = "Unassigned"
    best_bp = 0
    details = {}
    for label, chrom_map in idx.items():
        s_e = chrom_map.get(chrom)
        if s_e is None:
            details[label] = 0
            continue
        bp = overlapped_bp(s_e[0], s_e[1], int(start), int(end))
        details[label] = bp
        if bp > best_bp:
            best_bp = bp
            best_label = label
    return best_label, details


def read_significant_gene_regions(sig_dir: Path, sig_glob: str) -> pd.DataFrame:
    """Extract approximate gene regions from significant event results (merged by min/max coordinates per gene).
    Rules: gene key prefers geneSymbol; if missing, uses GeneID. Coordinates are taken as min/max of available columns per rMATS event type.
    Supported events: SE, RI, A3SS, A5SS, MXE.
    Returns columns: ['gene_key','geneSymbol','GeneID','chrom','start','end'].
    """
    paths = sorted(sig_dir.glob(sig_glob))
    frames = []

    # Event type to possible coordinate column mapping
    event_coord_cols = {
        "SE": [
            "exonStart_0base", "exonEnd",
            "upstreamES", "upstreamEE", "downstreamES", "downstreamEE",
        ],
        "RI": [
            "riExonStart_0base", "riExonEnd",
            "upstreamES", "upstreamEE", "downstreamES", "downstreamEE",
        ],
        "A3SS": [
            "longExonStart_0base", "longExonEnd",
            "shortES", "shortEE",
            "flankingES", "flankingEE",
        ],
        "A5SS": [
            "longExonStart_0base", "longExonEnd",
            "shortES", "shortEE",
            "flankingES", "flankingEE",
        ],
        "MXE": [
            "1stExonStart_0base", "1stExonEnd",
            "2ndExonStart_0base", "2ndExonEnd",
            "upstreamES", "upstreamEE", "downstreamES", "downstreamEE",
        ],
    }

    for p in paths:
        df = pd.read_csv(p, sep="\t", dtype=str)
        # Required base columns
        if "chr" not in df.columns:
            raise ValueError(f"File missing column 'chr': {p}")

        # Identify event type (prefer column value, then infer from filename)
        file_evt = None
        if "event_type" in df.columns:
            vals = pd.unique(df["event_type"].dropna().astype(str))
            if len(vals) == 1 and vals[0] in event_coord_cols:
                file_evt = vals[0]
        if file_evt is None:
            stem = Path(p).name.split(".")[0].upper()
            if stem in event_coord_cols:
                file_evt = stem

        # Aggregate available coordinate columns (mapped for this event type and present in the table)
        cand_cols = []
        if file_evt is not None:
            cand_cols = [c for c in event_coord_cols[file_evt] if c in df.columns]
        else:
            # If event type cannot be identified, select from all candidate columns that exist
            all_candidates = [c for cols in event_coord_cols.values() for c in cols]
            cand_cols = sorted(set(c for c in all_candidates if c in df.columns))

        if len(cand_cols) == 0:
            raise ValueError(f"No usable coordinate columns found: {p} available columns={list(df.columns)}")

        # Gene identifier
        gene_symbol = df.get("geneSymbol")
        gene_id = df.get("GeneID")
        key_series = gene_symbol.fillna("") if gene_symbol is not None else pd.Series([""]*len(df))
        fallback = gene_id.fillna("") if gene_id is not None else pd.Series([""]*len(df))
        gene_key = key_series.where(key_series.astype(str).str.len() > 0, fallback)

        # Row-level interval (take min/max of available columns)
        coords = df[cand_cols].applymap(_coerce_int)
        row_start = coords.min(axis=1)
        row_end = coords.max(axis=1)

        part = pd.DataFrame({
            "gene_key": gene_key.astype(str),
            "geneSymbol": (gene_symbol.astype(str) if gene_symbol is not None else ""),
            "GeneID": (gene_id.astype(str) if gene_id is not None else ""),
            "chrom": df["chr"].astype(str),
            "start": row_start.astype(int),
            "end": row_end.astype(int),
            "source_file": str(p)
        })
        # Filter out entries without gene_key
        part = part[part["gene_key"].astype(bool)]
        frames.append(part)

    all_rows = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["gene_key","geneSymbol","GeneID","chrom","start","end"])

    # Merge min/max coordinates per gene per chromosome (conservative approximate gene region)
    agg = (all_rows
           .groupby(["gene_key","chrom"], as_index=False)
           .agg({"start":"min","end":"max","geneSymbol":"first","GeneID":"first"}))

    # If the same gene spans multiple chromosomes (very rare), keep one entry per chromosome
    return agg.reset_index(drop=True)

In [ ]:
# ========== Read compartments and build index ==========
comp_frames = []
for label, path in COMPARTMENT_FILES.items():
    df = read_compartment_bed(path)
    df["label"] = label  # Override with mapping key to avoid inconsistency with the 4th column in the file
    comp_frames.append(df)
comp_df = pd.concat(comp_frames, ignore_index=True)
comp_index = build_interval_index(comp_df)
print("Number of compartment segments:", len(comp_df))

In [ ]:
# ========== Read significant genes and build gene regions ==========
gene_regions = read_significant_gene_regions(SIG_DIR, SIG_GLOB)
print("Gene entries after deduplication:", len(gene_regions))

gene_regions.head(3)

In [ ]:
# ========== Assign compartment to each gene ==========
assign_rows = []
for _, r in gene_regions.iterrows():
    best, detail = assign_compartment(r["chrom"], int(r["start"]), int(r["end"]), comp_index)
    row = {
        "gene_key": r["gene_key"],
        "geneSymbol": r.get("geneSymbol", None),
        "GeneID": r.get("GeneID", None),
        "chrom": r["chrom"],
        "start": int(r["start"]),
        "end": int(r["end"]),
        "assigned": best,
    }
    # Record overlap bp for each category
    for k, v in detail.items():
        row[f"overlap_{k}"] = int(v)
    assign_rows.append(row)

assign_df = pd.DataFrame(assign_rows)
assign_path = OUTPUT_DIR / "assigned_genes.tsv"
assign_df.to_csv(assign_path, sep="\t", index=False)
print("Saved:", assign_path)

# Summarize proportions
summary = assign_df["assigned"].value_counts(dropna=False).rename_axis("compartment").reset_index(name="count")
summary["proportion"] = summary["count"] / summary["count"].sum()
summary_path = OUTPUT_DIR / "proportions.tsv"
summary.to_csv(summary_path, sep="\t", index=False)
print("Saved:", summary_path)

summary

In [ ]:
# ========== Visualization ==========
plt.figure(figsize=(5, 4))
order = COMPARTMENT_ORDER
plot_df = summary.set_index("compartment").reindex(order).reset_index().dropna(subset=["count"])  

# Use fixed color palette to ensure consistent group colors
ax = sns.barplot(
    data=plot_df,
    x="compartment",
    y="proportion",
    order=order,
    palette=COMPARTMENT_COLORS,
)

# Percentage labels
for i, v in enumerate(plot_df["proportion"].values):
    ax.text(i, v + 0.01, f"{v:.2%}", ha="center", va="bottom", fontsize=10)

plt.ylim(0, max(0.01, plot_df["proportion"].max() * 1.15))
plt.ylabel("Proportion of genes")
plt.title("Splicing-significant genes in A/B compartments")

# Custom legend: only show groups that are present
legend_labels = [c for c in ["StableA", "StableB", "A2B", "B2A"] if c in plot_df["compartment"].values]
handles = [Patch(facecolor=COMPARTMENT_COLORS[c], label=c) for c in legend_labels]
ax.legend(handles=handles, title=None, frameon=False, loc="upper right")

plt.tight_layout()

fig_path = OUTPUT_DIR / "ab_compartment_proportions.pdf"
plt.savefig(fig_path, dpi=300)
plt.show()
print("Figure saved:", fig_path)